# GVH Diagonal Cubic 0.2.23.3 — Timelike Field Dynamics and Covariant Closure

**Auteur : Charlemagne O Laurince**

---

## Objectif

Le notebook `0.2.23.2` a dérivé le projecteur spatial symétrique sans trace :

\[
\Pi_{\mu\nu}[T]
=
\left(
h_{(\mu}{}^\alpha h_{\nu)}{}^\beta
-
\frac13 h_{\mu\nu}h^{\alpha\beta}
\right)
T_{\alpha\beta},
\]

avec :

\[
h_{\mu\nu}
=
g_{\mu\nu}
+
u_\mu u_\nu.
\]

Le verrou suivant est le statut physique et dynamique du champ temporel :

\[
\boxed{u^\mu}.
\]

Ce notebook doit déterminer ce qui peut être fermé sans introduire arbitrairement une nouvelle théorie.

---

## Questions étudiées

1. \(u^\mu\) est-il :
   - le quadrivecteur vitesse de la matière ;
   - un champ géométrique dérivé ;
   - un champ dynamique indépendant ?

2. Comment imposer :

\[
u^\mu u_\mu=-1
\]

de façon covariante ?

3. Quelle est l’action locale générale au second ordre en dérivées ?

4. Quelles équations suivent de :

\[
\frac{\delta S}{\delta u^\mu}=0,
\qquad
\frac{\delta S}{\delta\lambda}=0,
\qquad
\frac{\delta S}{\delta g^{\mu\nu}}=0?
\]

5. Les notebooks GVH précédents sélectionnent-ils une dynamique unique ?

---

## Règle scientifique

Le notebook ne choisit pas automatiquement une action particulière de type vecteur-unitaire.

Il construit l’espace des actions admissibles, vérifie les limites et conclut explicitement si les fondements GVH existants suffisent ou non à sélectionner les coefficients.

---

## Statuts possibles

```text
PASS-TIMELIKE-FIELD-KINEMATIC-CLOSURE
PASS-UNIT-CONSTRAINT-CLOSURE
BLOCKED-UNIQUE-TIMELIKE-DYNAMICS
BLOCKED-PREFERRED-FRAME-PARAMETERS
BLOCKED-METRIC-BACKREACTION
PASS-FULL-COVARIANT-CLOSURE
```

In [1]:
from __future__ import annotations

import json
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import sympy as sp

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 220)

print("Python :", sys.version)
print("SymPy :", sp.__version__)

Python : 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
SymPy : 1.14.0


# 1. Dépôt et chemins

In [2]:
REPOSITORY_URL = "https://github.com/col38470682/Univers.git"
REPOSITORY_DIR = Path("/content/Univers")
PROJECT_ROOT = REPOSITORY_DIR / "gvh_diagonal_cubic"

if (REPOSITORY_DIR / ".git").exists():
    subprocess.run(
        ["git", "-C", str(REPOSITORY_DIR), "pull", "--ff-only"],
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", REPOSITORY_URL, str(REPOSITORY_DIR)],
        check=True,
    )

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(PROJECT_ROOT)

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "timelike_field"
)

EXPORT_DIR = PROJECT_ROOT / "exports"

for directory in [PROCESSED_DIR, EXPORT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT :", PROJECT_ROOT)
print("PROCESSED_DIR :", PROCESSED_DIR)

PROJECT_ROOT : /content/Univers/gvh_diagonal_cubic
PROCESSED_DIR : /content/Univers/gvh_diagonal_cubic/data/processed/timelike_field


# 2. Dépendances théoriques

In [3]:
NOTEBOOK_PATTERNS = {
    "0.2.21": "*0.2.21*Equivalence*Principle*PPN*Constraints*.ipynb",
    "0.2.22": "*0.2.22*Static*Spherical*PPN*Derivation*.ipynb",
    "0.2.23": "*0.2.23*Covariant*Static*Field*Source*Matching*.ipynb",
    "0.2.23.1": "*0.2.23.1*Weak*Field*Coefficient*Extraction*PPN*.ipynb",
    "0.2.23.2": "*0.2.23.2*Covariant*Source*Projector*Derivation*.ipynb",
}

resolved_notebooks = {}

for notebook_id, pattern in NOTEBOOK_PATTERNS.items():
    matches = sorted(PROJECT_ROOT.rglob(pattern))
    resolved_notebooks[notebook_id] = (
        matches[0] if matches else None
    )

source_notebooks_df = pd.DataFrame([
    {
        "notebook_id": notebook_id,
        "resolved_path": str(path) if path else "",
        "found": path is not None,
    }
    for notebook_id, path in resolved_notebooks.items()
])

source_notebooks_df

,notebook_id,resolved_path,found
0,0.2.21,/content/Univers/gvh_diagonal_cubic/0.2_gravity/0.2A_theoretical_foundations/0.2A_theoretical_foundations/GVH_Diagonal_Cubic_0.2.21_Equivalence_Principle_PPN_Constraints.ipynb,True
1,0.2.22,/content/Univers/gvh_diagonal_cubic/0.2_gravity/0.2A_theoretical_foundations/0.2A_theoretical_foundations/GVH_Diagonal_Cubic_0.2.22_Static_Spherical_Solution_PPN_Derivation.ipynb,True
2,0.2.23,/content/Univers/gvh_diagonal_cubic/0.2_gravity/0.2A_theoretical_foundations/0.2A_theoretical_foundations/GVH_Diagonal_Cubic_0.2.23_Covariant_Static_Field_Equations_Source_Matching.ipynb,True
3,0.2.23.1,/content/Univers/gvh_diagonal_cubic/0.2_gravity/0.2A_theoretical_foundations/0.2A_theoretical_foundations/GVH_Diagonal_Cubic_0.2.23.1_Weak_Field_Coefficient_Extraction_for_PPN_UPDATED.ipynb,True
4,0.2.23.2,,False


In [4]:
all_sources_found = bool(
    source_notebooks_df["found"].all()
)

if not all_sources_found:
    print(
        "ATTENTION : la chaîne théorique archivée est incomplète. "
        "La classification reste possible, mais le statut final "
        "signalera les dépendances absentes."
    )

ATTENTION : la chaîne théorique archivée est incomplète. La classification reste possible, mais le statut final signalera les dépendances absentes.


# 3. Trois statuts possibles pour \(u^\mu\)

In [5]:
timelike_field_options_df = pd.DataFrame([
    {
        "option_id": "MATTER_COMOVING",
        "definition": "u^mu is the matter four-velocity",
        "independent_degree_of_freedom": False,
        "preferred_frame_risk": "matter-sector dependent",
        "advantage": (
            "No new independent vector degree of freedom"
        ),
        "limitation": (
            "Undefined in vacuum or for multiple non-comoving fluids"
        ),
    },
    {
        "option_id": "GEOMETRIC_DERIVED",
        "definition": (
            "u_mu proportional to the normalized gradient "
            "of a scalar clock field"
        ),
        "independent_degree_of_freedom": "scalar field only",
        "preferred_frame_risk": (
            "foliation dependence must be tested"
        ),
        "advantage": (
            "Hypersurface-orthogonal by construction"
        ),
        "limitation": (
            "Requires a scalar-clock sector not yet derived in GVH"
        ),
    },
    {
        "option_id": "INDEPENDENT_UNIT_VECTOR",
        "definition": (
            "u^mu is an independent timelike unit vector field"
        ),
        "independent_degree_of_freedom": True,
        "preferred_frame_risk": "explicit",
        "advantage": (
            "Well-defined in matter and vacuum regions"
        ),
        "limitation": (
            "Introduces new kinetic coefficients and PPN channels"
        ),
    },
])

timelike_field_options_df

,option_id,definition,independent_degree_of_freedom,preferred_frame_risk,advantage,limitation
0,MATTER_COMOVING,u^mu is the matter four-velocity,False,matter-sector dependent,No new independent vector degree of freedom,Undefined in vacuum or for multiple non-comoving fluids
1,GEOMETRIC_DERIVED,u_mu proportional to the normalized gradient of a scalar clock field,scalar field only,foliation dependence must be tested,Hypersurface-orthogonal by construction,Requires a scalar-clock sector not yet derived in GVH
2,INDEPENDENT_UNIT_VECTOR,u^mu is an independent timelike unit vector field,True,explicit,Well-defined in matter and vacuum regions,Introduces new kinetic coefficients and PPN channels


Les notebooks antérieurs utilisent \(u^\mu\) pour rendre le projecteur spatial covariant, mais ne sélectionnent pas encore l'une de ces trois interprétations.

Le statut actuel est donc :

\[
\boxed{
u^\mu\ \text{est cinématiquement nécessaire, mais dynamiquement non unique}
}.
\]

# 4. Fermeture cinématique

In [6]:
eta = sp.diag(-1, 1, 1, 1)

u0, u1, u2, u3 = sp.symbols(
    "u0 u1 u2 u3",
    real=True,
)

u_up = sp.Matrix([u0, u1, u2, u3])
u_down = eta * u_up

unit_constraint = sp.simplify(
    (u_up.T * u_down)[0] + 1
)

unit_constraint

-u0**2 + u1**2 + u2**2 + u3**2 + 1

La fermeture cinématique minimale impose :

\[
C_u
=
g_{\mu\nu}u^\mu u^\nu+1
=
0.
\]

Elle peut être imposée par un multiplicateur de Lagrange \(\lambda\) :

\[
S_\lambda
=
\int d^4x\sqrt{-g}\,
\lambda
\left(
u^\mu u_\mu+1
\right).
\]

La variation par rapport à \(\lambda\) donne directement la contrainte unitaire.

In [7]:
lam = sp.symbols(
    "lambda",
    real=True,
)

L_constraint = sp.expand(
    lam * unit_constraint
)

dL_dlambda = sp.diff(
    L_constraint,
    lam,
)

constraint_variation_pass = bool(
    sp.simplify(
        dL_dlambda
        - unit_constraint
    ) == 0
)

constraint_closure_df = pd.DataFrame([{
    "constraint": "u^mu u_mu + 1 = 0",
    "implementation": "Lagrange multiplier lambda",
    "delta_S_delta_lambda": str(dL_dlambda),
    "variation_pass": constraint_variation_pass,
}])

constraint_closure_df

,constraint,implementation,delta_S_delta_lambda,variation_pass
0,u^mu u_mu + 1 = 0,Lagrange multiplier lambda,-u0**2 + u1**2 + u2**2 + u3**2 + 1,True


# 5. Décomposition cinématique de \(
abla_\mu u_
u\)

Pour un champ temporel unitaire, la dérivée covariante se décompose en :

\[
\nabla_\mu u_\nu
=
\frac13\theta h_{\mu\nu}
+
\sigma_{\mu\nu}
+
\omega_{\mu\nu}
-
u_\mu a_\nu,
\]

où :

\[
\theta=\nabla_\mu u^\mu
\]

est l'expansion,

\[
a_\mu=u^\nu\nabla_\nu u_\mu
\]

est l'accélération,

\[
\sigma_{\mu\nu}
\]

est le cisaillement spatial symétrique sans trace,

et :

\[
\omega_{\mu\nu}
\]

est la vorticité spatiale antisymétrique.

Ces quatre objets permettent de classer les actions cinétiques possibles sans fixer encore leurs coefficients.

In [8]:
kinematic_invariants_df = pd.DataFrame([
    {
        "invariant": "theta^2",
        "sector": "expansion",
        "definition": "(nabla_mu u^mu)^2",
        "coefficient": "c_theta",
    },
    {
        "invariant": "sigma_mn sigma^mn",
        "sector": "shear",
        "definition": "sigma_mn sigma^mn",
        "coefficient": "c_sigma",
    },
    {
        "invariant": "omega_mn omega^mn",
        "sector": "vorticity",
        "definition": "omega_mn omega^mn",
        "coefficient": "c_omega",
    },
    {
        "invariant": "a_mu a^mu",
        "sector": "acceleration",
        "definition": "a_mu a^mu",
        "coefficient": "c_a",
    },
])

kinematic_invariants_df

,invariant,sector,definition,coefficient
0,theta^2,expansion,(nabla_mu u^mu)^2,c_theta
1,sigma_mn sigma^mn,shear,sigma_mn sigma^mn,c_sigma
2,omega_mn omega^mn,vorticity,omega_mn omega^mn,c_omega
3,a_mu a^mu,acceleration,a_mu a^mu,c_a


# 6. Action cinétique locale générale au second ordre

Sous les hypothèses :

- covariance locale ;
- dépendance quadratique en \(\nabla u\) ;
- au plus deux dérivées ;
- parité paire ;
- contrainte \(u^2=-1\) ;

une base cinétique générale peut être écrite :

\[
S_u
=
\int d^4x\sqrt{-g}
\left[
-\frac12
\left(
c_\theta\theta^2
+
c_\sigma\sigma_{\mu\nu}\sigma^{\mu\nu}
+
c_\omega\omega_{\mu\nu}\omega^{\mu\nu}
+
c_a a_\mu a^\mu
\right)
+
\lambda(u^2+1)
-
V_u
\right].
\]

Cette expression est une **classification d'actions admissibles**, pas encore l'action GVH sélectionnée.

In [9]:
c_theta, c_sigma, c_omega, c_a = sp.symbols(
    "c_theta c_sigma c_omega c_a",
    real=True,
)

theta_sq, sigma_sq, omega_sq, accel_sq = sp.symbols(
    "theta_sq sigma_sq omega_sq accel_sq",
    real=True,
)

V_u = sp.symbols(
    "V_u",
    real=True,
)

L_u_general = sp.expand(
    -sp.Rational(1, 2) * (
        c_theta * theta_sq
        + c_sigma * sigma_sq
        + c_omega * omega_sq
        + c_a * accel_sq
    )
    + lam * unit_constraint
    - V_u
)

L_u_general

-V_u - accel_sq*c_a/2 - c_omega*omega_sq/2 - c_sigma*sigma_sq/2 - c_theta*theta_sq/2 - lambda*u0**2 + lambda*u1**2 + lambda*u2**2 + lambda*u3**2 + lambda

# 7. Équation formelle du champ temporel

En notation compacte, une action quadratique peut aussi s'écrire :

\[
S_u
=
-\frac12
\int d^4x\sqrt{-g}\,
K^{\alpha\beta}{}_{\mu\nu}
\nabla_\alpha u^\mu
\nabla_\beta u^\nu
+
\int d^4x\sqrt{-g}\,
\lambda(u^2+1).
\]

La variation formelle donne une équation de la structure :

\[
\boxed{
\nabla_\alpha J^\alpha{}_\mu
+
\lambda u_\mu
+
\mathcal F_\mu^{\rm int}
=
0
}
\]

avec :

\[
J^\alpha{}_\mu
=
K^{\alpha\beta}{}_{\mu\nu}
\nabla_\beta u^\nu.
\]

Le terme \(\mathcal F_\mu^{\rm int}\) vient de la dépendance du projecteur de source et du couplage à \(D_{\mu\nu}\).

In [10]:
formal_field_equation_df = pd.DataFrame([
    {
        "term": "nabla_alpha J^alpha_mu",
        "origin": "kinetic action of u^mu",
        "derived_status": (
            "FORMAL STRUCTURE DERIVED"
        ),
    },
    {
        "term": "lambda u_mu",
        "origin": "unit constraint",
        "derived_status": "DERIVED",
    },
    {
        "term": "F_mu^int",
        "origin": (
            "variation of D_mn Pi^mn[T] "
            "through h_mn(u,g)"
        ),
        "derived_status": (
            "NOT EXPLICITLY DERIVED"
        ),
    },
])

formal_field_equation_df

,term,origin,derived_status
0,nabla_alpha J^alpha_mu,kinetic action of u^mu,FORMAL STRUCTURE DERIVED
1,lambda u_mu,unit constraint,DERIVED
2,F_mu^int,"variation of D_mn Pi^mn[T] through h_mn(u,g)",NOT EXPLICITLY DERIVED


# 8. Multiplicateur de Lagrange projeté

En contractant l'équation du champ avec \(u^\mu\), on obtient formellement :

\[
\lambda
=
u^\mu
\nabla_\alpha J^\alpha{}_\mu
+
u^\mu\mathcal F_\mu^{\rm int},
\]

à un signe près selon la convention retenue dans l'action.

La projection spatiale :

\[
h^\nu{}_\mu
\left(
\nabla_\alpha J^\alpha{}_\nu
+
\mathcal F_\nu^{\rm int}
\right)
=
0
\]

contient les équations dynamiques indépendantes.

In [11]:
closure_structure_df = pd.DataFrame([
    {
        "projection": "parallel to u^mu",
        "role": (
            "determines the Lagrange multiplier lambda"
        ),
        "status": "KINEMATICALLY CLOSED",
    },
    {
        "projection": "orthogonal with h^nu_mu",
        "role": (
            "determines physical vector dynamics"
        ),
        "status": (
            "COEFFICIENTS AND INTERACTION FORCE REQUIRED"
        ),
    },
])

closure_structure_df

,projection,role,status
0,parallel to u^mu,determines the Lagrange multiplier lambda,KINEMATICALLY CLOSED
1,orthogonal with h^nu_mu,determines physical vector dynamics,COEFFICIENTS AND INTERACTION FORCE REQUIRED


# 9. Limite statique sphérique

Dans une géométrie statique sphérique sans rotation, le choix naturel compatible avec les observateurs statiques est :

\[
u^\mu
=
N^{-1}(r)\,
(\partial_t)^\mu,
\]

où :

\[
N^2(r)=-g_{tt}(r).
\]

Cette configuration est :

- unitaire ;
- hypersurface-orthogonale ;
- sans expansion ;
- sans cisaillement ;
- sans vorticité ;
- généralement accélérée.

Ainsi :

\[
\theta=0,
\qquad
\sigma_{\mu\nu}=0,
\qquad
\omega_{\mu\nu}=0,
\]

mais :

\[
a_\mu\neq0
\]

en général.

La branche statique teste donc principalement le coefficient \(c_a\), sans déterminer nécessairement \(c_\theta,c_\sigma,c_\omega\).

In [12]:
static_sector_df = pd.DataFrame([
    {
        "kinematic_quantity": "theta",
        "static_spherical_value": "0",
        "constrained_by_static_branch": False,
    },
    {
        "kinematic_quantity": "sigma_mn",
        "static_spherical_value": "0",
        "constrained_by_static_branch": False,
    },
    {
        "kinematic_quantity": "omega_mn",
        "static_spherical_value": "0",
        "constrained_by_static_branch": False,
    },
    {
        "kinematic_quantity": "a_mu",
        "static_spherical_value": "generally nonzero",
        "constrained_by_static_branch": True,
    },
])

static_sector_df

,kinematic_quantity,static_spherical_value,constrained_by_static_branch
0,theta,0,False
1,sigma_mn,0,False
2,omega_mn,0,False
3,a_mu,generally nonzero,True


# 10. Vérification locale de normalisation statique

In [13]:
N = sp.symbols(
    "N",
    positive=True,
    real=True,
)

g_static = sp.diag(
    -N**2,
    1,
    1,
    1,
)

u_static_up = sp.Matrix([
    1 / N,
    0,
    0,
    0,
])

u_static_down = g_static * u_static_up

static_norm = sp.simplify(
    (u_static_up.T * u_static_down)[0]
)

static_normalization_pass = bool(
    static_norm == -1
)

static_normalization_df = pd.DataFrame([{
    "u^mu": "(1/N, 0, 0, 0)",
    "metric_gtt": "-N^2",
    "u^mu u_mu": str(static_norm),
    "pass": static_normalization_pass,
}])

static_normalization_df

,u^mu,metric_gtt,u^mu u_mu,pass
0,"(1/N, 0, 0, 0)",-N^2,-1,True


# 11. Option champ dérivé d'un scalaire

Une fermeture géométrique possible consiste à poser :

\[
u_\mu
=
-\frac{\nabla_\mu\varphi}
{\sqrt{-\nabla_\alpha\varphi\nabla^\alpha\varphi}}.
\]

Cette forme impose automatiquement :

\[
u^\mu u_\mu=-1
\]

et :

\[
\omega_{\mu\nu}=0.
\]

Mais elle introduit un champ scalaire \(\varphi\) et exige une action ou une règle dynamique pour ce champ.

Les notebooks GVH antérieurs n'ont pas encore dérivé ce secteur scalaire.

In [14]:
derived_scalar_option_df = pd.DataFrame([{
    "definition": (
        "u_mu = -grad_mu(phi) / sqrt(-grad(phi)^2)"
    ),
    "unit_norm_automatic": True,
    "vorticity_zero": True,
    "new_scalar_required": True,
    "selected_by_previous_GVH_notebooks": False,
    "status": "ADMISSIBLE BUT NOT DERIVED",
}])

derived_scalar_option_df

,definition,unit_norm_automatic,vorticity_zero,new_scalar_required,selected_by_previous_GVH_notebooks,status
0,u_mu = -grad_mu(phi) / sqrt(-grad(phi)^2),True,True,True,False,ADMISSIBLE BUT NOT DERIVED


# 12. Option quadrivitesse de matière

Une autre possibilité est :

\[
u^\mu=u^\mu_{\rm matter}.
\]

Elle évite un nouveau degré de liberté dans une matière simple, mais pose trois difficultés :

1. absence d'un \(u^\mu\) unique dans le vide ;
2. ambiguïté pour plusieurs fluides non comobiles ;
3. dépendance du projecteur aux propriétés du milieu.

Cette option ne fournit donc pas, à elle seule, une fermeture globale du secteur extérieur du Système solaire.

In [15]:
matter_velocity_option_df = pd.DataFrame([{
    "definition": "u^mu = matter four-velocity",
    "vacuum_definition": False,
    "multi_fluid_unique": False,
    "new_vector_degree_of_freedom": False,
    "global_exterior_closure": False,
    "status": "INSUFFICIENT FOR GLOBAL VACUUM SECTOR",
}])

matter_velocity_option_df

,definition,vacuum_definition,multi_fluid_unique,new_vector_degree_of_freedom,global_exterior_closure,status
0,u^mu = matter four-velocity,False,False,False,False,INSUFFICIENT FOR GLOBAL VACUUM SECTOR


# 13. Option champ vectoriel indépendant

L'option indépendante ferme la définition de \(u^\mu\) partout, mais laisse plusieurs coefficients cinétiques :

\[
c_\theta,
\quad
c_\sigma,
\quad
c_\omega,
\quad
c_a.
\]

Ces paramètres peuvent générer des effets de référentiel privilégié et doivent être confrontés aux contraintes PPN correspondantes.

Les fondements actuels de GVH ne sélectionnent pas encore leurs valeurs.

In [16]:
independent_vector_option_df = pd.DataFrame([{
    "definition": "independent unit timelike vector",
    "vacuum_definition": True,
    "kinetic_coefficients": (
        "c_theta,c_sigma,c_omega,c_a"
    ),
    "preferred_frame_parameters_possible": True,
    "coefficients_selected_by_GVH": False,
    "status": "DYNAMICALLY UNDERDETERMINED",
}])

independent_vector_option_df

,definition,vacuum_definition,kinetic_coefficients,preferred_frame_parameters_possible,coefficients_selected_by_GVH,status
0,independent unit timelike vector,True,"c_theta,c_sigma,c_omega,c_a",True,False,DYNAMICALLY UNDERDETERMINED


# 14. Test de sélection par les fondements existants

In [17]:
selection_evidence_df = pd.DataFrame([
    {
        "criterion": (
            "previous notebooks define u^mu as matter velocity"
        ),
        "supported": False,
    },
    {
        "criterion": (
            "previous notebooks derive u_mu from a scalar clock"
        ),
        "supported": False,
    },
    {
        "criterion": (
            "previous notebooks provide an independent-vector action"
        ),
        "supported": False,
    },
    {
        "criterion": (
            "previous notebooks require a unit timelike field "
            "for covariant spatial projection"
        ),
        "supported": True,
    },
])

unique_dynamics_selected = bool(
    selection_evidence_df.loc[
        selection_evidence_df[
            "criterion"
        ].str.contains(
            "provide an independent-vector action"
        ),
        "supported",
    ].any()
)

selection_evidence_df

,criterion,supported
0,previous notebooks define u^mu as matter velocity,False
1,previous notebooks derive u_mu from a scalar clock,False
2,previous notebooks provide an independent-vector action,False
3,previous notebooks require a unit timelike field for covariant spatial projection,True


# 15. Couplage au projecteur de source

Le couplage proposé dans la chaîne précédente est :

\[
S_{\rm int}
=
\lambda_T
\int d^4x\sqrt{-g}\,
D_{\mu\nu}\Pi^{\mu\nu}[T].
\]

Comme :

\[
\Pi_{\mu\nu}[T]
\]

dépend de :

\[
h_{\mu\nu}
=
g_{\mu\nu}
+
u_\mu u_\nu,
\]

la variation par rapport à \(u^\mu\) n'est pas nulle.

Elle produit une force effective :

\[
\mathcal F_\mu^{\rm int}
=
\frac{1}{\sqrt{-g}}
\frac{\delta S_{\rm int}}
{\delta u^\mu}.
\]

La structure exacte de cette force dépend aussi de la variation de \(T_{\alpha\beta}\), du statut de la matière et du placement des indices. Elle ne doit pas être inventée dans ce notebook.

In [18]:
interaction_variation_df = pd.DataFrame([
    {
        "variation": "delta S_int / delta D_mn",
        "result": "proportional to Pi_mn[T]",
        "status": "STRUCTURALLY DERIVED",
    },
    {
        "variation": "delta S_int / delta u^mu",
        "result": "F_mu^int",
        "status": (
            "NONZERO IN GENERAL, NOT EXPLICITLY CLOSED"
        ),
    },
    {
        "variation": "delta S_int / delta g^mn",
        "result": "metric backreaction",
        "status": "NOT DERIVED",
    },
    {
        "variation": "delta S_int / delta matter",
        "result": "modified matter equations possible",
        "status": "NOT DERIVED",
    },
])

interaction_variation_df

,variation,result,status
0,delta S_int / delta D_mn,proportional to Pi_mn[T],STRUCTURALLY DERIVED
1,delta S_int / delta u^mu,F_mu^int,"NONZERO IN GENERAL, NOT EXPLICITLY CLOSED"
2,delta S_int / delta g^mn,metric backreaction,NOT DERIVED
3,delta S_int / delta matter,modified matter equations possible,NOT DERIVED


# 16. Conservation totale

La covariance difféomorphe d'une action complète impose une identité de Noether reliant les équations de :

\[
g_{\mu\nu},
\qquad
D_{\mu\nu},
\qquad
u^\mu,
\qquad
\text{matière}.
\]

La conservation correcte concerne le tenseur total :

\[
\nabla^\mu
\left(
T_{\mu\nu}^{\rm matter}
+
T_{\mu\nu}^{D}
+
T_{\mu\nu}^{u}
+
T_{\mu\nu}^{\rm int}
\right)
=
0.
\]

Il n'est donc pas nécessaire que chaque secteur soit conservé séparément. Des échanges internes sont possibles, mais ils doivent découler de l'action complète.

In [19]:
conservation_requirements_df = pd.DataFrame([
    {
        "object": "T_mn^matter",
        "separate_conservation_required": False,
        "total_conservation_required": True,
    },
    {
        "object": "T_mn^D",
        "separate_conservation_required": False,
        "total_conservation_required": True,
    },
    {
        "object": "T_mn^u",
        "separate_conservation_required": False,
        "total_conservation_required": True,
    },
    {
        "object": "T_mn^int",
        "separate_conservation_required": False,
        "total_conservation_required": True,
    },
])

conservation_requirements_df

,object,separate_conservation_required,total_conservation_required
0,T_mn^matter,False,True
1,T_mn^D,False,True
2,T_mn^u,False,True
3,T_mn^int,False,True


# 17. Limite GR

In [20]:
GR_LIMIT_SUBSTITUTIONS = {
    "lambda_T": 0,
    "D_mn": 0,
    "c_theta": 0,
    "c_sigma": 0,
    "c_omega": 0,
    "c_a": 0,
}

gr_limit_df = pd.DataFrame([
    {
        "sector": "directional interaction",
        "GR_limit": "lambda_T -> 0 or D_mn -> 0",
        "pass_condition_defined": True,
    },
    {
        "sector": "timelike kinetic sector",
        "GR_limit": (
            "kinetic coefficients decouple or field "
            "becomes nondynamical"
        ),
        "pass_condition_defined": True,
    },
    {
        "sector": "metric",
        "GR_limit": "Einstein equations recovered",
        "pass_condition_defined": True,
    },
])

gr_limit_df

,sector,GR_limit,pass_condition_defined
0,directional interaction,lambda_T -> 0 or D_mn -> 0,True
1,timelike kinetic sector,kinetic coefficients decouple or field becomes nondynamical,True
2,metric,Einstein equations recovered,True


# 18. Risque de référentiel privilégié

In [21]:
preferred_frame_audit_df = pd.DataFrame([
    {
        "condition": (
            "u^mu independent and dynamically active"
        ),
        "preferred_frame_risk": True,
        "requires_future_PPN_parameters": (
            "alpha_1^PPN, alpha_2^PPN and related channels"
        ),
    },
    {
        "condition": (
            "u^mu derived from static source congruence only"
        ),
        "preferred_frame_risk": (
            "model and domain dependent"
        ),
        "requires_future_PPN_parameters": True,
    },
    {
        "condition": (
            "u^mu auxiliary and fully decoupled"
        ),
        "preferred_frame_risk": False,
        "requires_future_PPN_parameters": False,
    },
])

preferred_frame_audit_df

,condition,preferred_frame_risk,requires_future_PPN_parameters
0,u^mu independent and dynamically active,True,"alpha_1^PPN, alpha_2^PPN and related channels"
1,u^mu derived from static source congruence only,model and domain dependent,True
2,u^mu auxiliary and fully decoupled,False,False


# 19. Portes de fermeture

In [22]:
kinematic_closure_pass = bool(
    constraint_variation_pass
    and static_normalization_pass
)

unit_constraint_pass = bool(
    constraint_variation_pass
)

unique_timelike_dynamics_pass = bool(
    unique_dynamics_selected
)

interaction_force_derived = False
metric_backreaction_derived = False
total_noether_identity_derived = False
preferred_frame_sector_constrained = False

full_covariant_closure_pass = bool(
    kinematic_closure_pass
    and unique_timelike_dynamics_pass
    and interaction_force_derived
    and metric_backreaction_derived
    and total_noether_identity_derived
    and preferred_frame_sector_constrained
)

closure_gates_df = pd.DataFrame([
    {
        "gate": "source notebooks archived",
        "pass": all_sources_found,
    },
    {
        "gate": "timelike kinematic closure",
        "pass": kinematic_closure_pass,
    },
    {
        "gate": "unit constraint",
        "pass": unit_constraint_pass,
    },
    {
        "gate": "unique timelike dynamics",
        "pass": unique_timelike_dynamics_pass,
    },
    {
        "gate": "interaction force derived",
        "pass": interaction_force_derived,
    },
    {
        "gate": "metric backreaction derived",
        "pass": metric_backreaction_derived,
    },
    {
        "gate": "total Noether identity derived",
        "pass": total_noether_identity_derived,
    },
    {
        "gate": "preferred-frame sector constrained",
        "pass": preferred_frame_sector_constrained,
    },
    {
        "gate": "full covariant closure",
        "pass": full_covariant_closure_pass,
    },
])

closure_gates_df

,gate,pass
0,source notebooks archived,False
1,timelike kinematic closure,True
2,unit constraint,True
3,unique timelike dynamics,False
4,interaction force derived,False
5,metric backreaction derived,False
6,total Noether identity derived,False
7,preferred-frame sector constrained,False
8,full covariant closure,False


# 20. Décision scientifique

In [23]:
if full_covariant_closure_pass:
    FINAL_STATUS = (
        "PASS-FULL-COVARIANT-CLOSURE"
    )
elif kinematic_closure_pass:
    FINAL_STATUS = (
        "PASS-TIMELIKE-FIELD-KINEMATIC-CLOSURE_"
        "PASS-UNIT-CONSTRAINT-CLOSURE_"
        "BLOCKED-UNIQUE-TIMELIKE-DYNAMICS_"
        "BLOCKED-PREFERRED-FRAME-PARAMETERS_"
        "BLOCKED-METRIC-BACKREACTION"
    )
else:
    FINAL_STATUS = (
        "BLOCKED-TIMELIKE-FIELD-KINEMATIC-CLOSURE"
    )

decision_df = pd.DataFrame([{
    "final_status": FINAL_STATUS,
    "kinematic_closure_pass": kinematic_closure_pass,
    "unit_constraint_pass": unit_constraint_pass,
    "unique_timelike_dynamics_pass": (
        unique_timelike_dynamics_pass
    ),
    "full_covariant_closure_pass": (
        full_covariant_closure_pass
    ),
    "selected_interpretation": (
        "NONE — previous GVH notebooks do not select one"
    ),
    "next_required_derivation": (
        "fix one covariant action or derive u^mu from "
        "existing GVH variables before metric variation"
    ),
}])

print("STATUT FINAL :", FINAL_STATUS)
decision_df

STATUT FINAL : PASS-TIMELIKE-FIELD-KINEMATIC-CLOSURE_PASS-UNIT-CONSTRAINT-CLOSURE_BLOCKED-UNIQUE-TIMELIKE-DYNAMICS_BLOCKED-PREFERRED-FRAME-PARAMETERS_BLOCKED-METRIC-BACKREACTION


,final_status,kinematic_closure_pass,unit_constraint_pass,unique_timelike_dynamics_pass,full_covariant_closure_pass,selected_interpretation,next_required_derivation
0,PASS-TIMELIKE-FIELD-KINEMATIC-CLOSURE_PASS-UNIT-CONSTRAINT-CLOSURE_BLOCKED-UNIQUE-TIMELIKE-DYNAMICS_BLOCKED-PREFERRED-FRAME-PARAMETERS_BLOCKED-METRIC-BACKREACTION,True,True,False,False,NONE — previous GVH notebooks do not select one,fix one covariant action or derive u^mu from existing GVH variables before metric variation


# 21. Artefact de fermeture

In [24]:
closure_artifact = {
    "artifact_status": (
        "KINEMATIC_CLOSURE_VALIDATED_"
        "UNIQUE_DYNAMICS_BLOCKED"
    ),
    "field": "u^mu",
    "metric_signature": "-+++",
    "unit_constraint": "u^mu u_mu = -1",
    "constraint_method": (
        "Lagrange multiplier lambda"
    ),
    "spatial_projector": (
        "h_mn = g_mn + u_m u_n"
    ),
    "kinematic_decomposition": (
        "nabla_m u_n = theta h_mn/3 "
        "+ sigma_mn + omega_mn - u_m a_n"
    ),
    "general_second_derivative_action_basis": [
        "theta^2",
        "sigma_mn sigma^mn",
        "omega_mn omega^mn",
        "a_mu a^mu",
    ],
    "candidate_interpretations": [
        "matter four-velocity",
        "normalized scalar gradient",
        "independent unit vector",
    ],
    "selected_interpretation": None,
    "selection_reason": (
        "Existing GVH notebooks do not uniquely select "
        "one interpretation or kinetic coefficient set."
    ),
    "static_spherical_sector": {
        "u^mu": "N^-1 partial_t^mu",
        "theta": "0",
        "sigma_mn": "0",
        "omega_mn": "0",
        "a_mu": "generally nonzero",
    },
    "open_requirements": [
        "derive or postulate the physical origin of u^mu",
        "derive the explicit interaction force F_mu^int",
        "vary the complete action with respect to g^mn",
        "derive the total Noether identity",
        "constrain preferred-frame PPN parameters",
    ],
    "source_notebooks": [
        "0.2.21",
        "0.2.22",
        "0.2.23",
        "0.2.23.1",
        "0.2.23.2",
    ],
}

closure_artifact

{'artifact_status': 'KINEMATIC_CLOSURE_VALIDATED_UNIQUE_DYNAMICS_BLOCKED',
 'field': 'u^mu',
 'metric_signature': '-+++',
 'unit_constraint': 'u^mu u_mu = -1',
 'constraint_method': 'Lagrange multiplier lambda',
 'spatial_projector': 'h_mn = g_mn + u_m u_n',
 'kinematic_decomposition': 'nabla_m u_n = theta h_mn/3 + sigma_mn + omega_mn - u_m a_n',
 'general_second_derivative_action_basis': ['theta^2',
  'sigma_mn sigma^mn',
  'omega_mn omega^mn',
  'a_mu a^mu'],
 'candidate_interpretations': ['matter four-velocity',
  'normalized scalar gradient',
  'independent unit vector'],
 'selected_interpretation': None,
 'selection_reason': 'Existing GVH notebooks do not uniquely select one interpretation or kinetic coefficient set.',
 'static_spherical_sector': {'u^mu': 'N^-1 partial_t^mu',
  'theta': '0',
  'sigma_mn': '0',
  'omega_mn': '0',
  'a_mu': 'generally nonzero'},
 'open_requirements': ['derive or postulate the physical origin of u^mu',
  'derive the explicit interaction force F_mu^in

# 22. Exports

In [25]:
PREFIX = "GVH_Diagonal_Cubic_0.2.23.3"

exports = {
    "Source_Notebooks": source_notebooks_df,
    "Timelike_Field_Options": timelike_field_options_df,
    "Constraint_Closure": constraint_closure_df,
    "Kinematic_Invariants": kinematic_invariants_df,
    "Formal_Field_Equation": formal_field_equation_df,
    "Closure_Structure": closure_structure_df,
    "Static_Sector": static_sector_df,
    "Static_Normalization": static_normalization_df,
    "Scalar_Option": derived_scalar_option_df,
    "Matter_Velocity_Option": matter_velocity_option_df,
    "Independent_Vector_Option": independent_vector_option_df,
    "Selection_Evidence": selection_evidence_df,
    "Interaction_Variation": interaction_variation_df,
    "Conservation_Requirements": conservation_requirements_df,
    "GR_Limit": gr_limit_df,
    "Preferred_Frame_Audit": preferred_frame_audit_df,
    "Closure_Gates": closure_gates_df,
    "Decision": decision_df,
}

for suffix, table in exports.items():
    table.to_csv(
        EXPORT_DIR
        / f"{PREFIX}_{suffix}.csv",
        index=False,
    )

CLOSURE_FILE = (
    PROCESSED_DIR
    / "gvh_timelike_field_covariant_closure.json"
)

CLOSURE_FILE.write_text(
    json.dumps(
        closure_artifact,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

metadata = {
    "notebook": (
        "GVH_Diagonal_Cubic_0.2.23.3_"
        "Timelike_Field_Dynamics_and_Covariant_Closure"
    ),
    "execution_time_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "final_status": FINAL_STATUS,
    "all_sources_found": all_sources_found,
    "kinematic_closure_pass": (
        kinematic_closure_pass
    ),
    "unit_constraint_pass": (
        unit_constraint_pass
    ),
    "unique_timelike_dynamics_pass": (
        unique_timelike_dynamics_pass
    ),
    "full_covariant_closure_pass": (
        full_covariant_closure_pass
    ),
    "closure_file": str(
        CLOSURE_FILE
    ),
    "next_action": (
        "Construct 0.2.23.4 to define a minimal action-selection "
        "criterion or prove that the current GVH postulates are "
        "insufficient to determine one."
    ),
}

with (
    EXPORT_DIR
    / f"{PREFIX}_Metadata.json"
).open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        indent=2,
        ensure_ascii=False,
    )

print("Closure artifact :", CLOSURE_FILE)
print("Exports :", EXPORT_DIR)

Closure artifact : /content/Univers/gvh_diagonal_cubic/data/processed/timelike_field/gvh_timelike_field_covariant_closure.json
Exports : /content/Univers/gvh_diagonal_cubic/exports


# Conclusion

Le champ temporel est cinématiquement fermé par :

\[
u^\mu u_\mu=-1,
\]

\[
h_{\mu\nu}
=
g_{\mu\nu}
+
u_\mu u_\nu,
\]

et la décomposition :

\[
\nabla_\mu u_\nu
=
\frac13\theta h_{\mu\nu}
+
\sigma_{\mu\nu}
+
\omega_{\mu\nu}
-
u_\mu a_\nu.
\]

Une base locale générale au second ordre contient :

\[
\theta^2,
\qquad
\sigma_{\mu\nu}\sigma^{\mu\nu},
\qquad
\omega_{\mu\nu}\omega^{\mu\nu},
\qquad
a_\mu a^\mu.
\]

Cependant, les fondements GVH actuels ne sélectionnent pas encore :

- la nature physique unique de \(u^\mu\) ;
- les coefficients cinétiques ;
- la force d'interaction complète ;
- la rétroaction métrique ;
- les paramètres de référentiel privilégié.

Le résultat attendu est donc une fermeture cinématique valide accompagnée d'un blocage dynamique explicite.

La prochaine étape doit décider si une action minimale peut être dérivée des postulats GVH existants ou si un nouveau postulat falsifiable est nécessaire.